# DBoW3 / 词袋向量 检索实验

**两种方案对比**：
1. **ORB + 预训练词典**：使用 `orbvoc.dbow3`，需 pyDBoW3（通常从源码编译）
2. **SIFT + 自建词典**：SIFT 无通用预训练词典，需从 DB 图像采样用 KMeans 自建

**注意**：算指标时排除 industry2 和 rural2（与 HNSW_NetVLAD_analysis_cosine 一致）。

**依赖**：`opencv-contrib-python`（ORB、SIFT）、`scikit-learn`（SIFT 用）、`pyDBoW3`（ORB 用，可选）

In [5]:
import sys
from pathlib import Path

_nb_dir = Path().resolve()
if _nb_dir.name == "hnsw_performance_analysis":
    sys.path.insert(0, str(_nb_dir.parent))

import numpy as np
import time

from demo_readH5 import load_multi_netvlad_descriptors, DB_H5_PATHS, QUERY_H5_PATHS
from scene_config import get_db_scene_ranges, get_query_trajectory_ranges, DATASETS_BASE
from coords_gt_utils import build_gt_9_for_all_trajectories
from bow_retrieval import (
    get_all_image_paths,
    extract_sift_features,
    extract_orb_features,
    build_vocabulary,
    compute_bow_histogram,
    compute_tfidf_vectors,
    BoWDatabase,
    DBoW3Database,
    HAS_PYDBOW3,
)

## 1. 加载配置与图像路径

In [6]:
# 从 H5 获取 db_names / query_names（与 NetVLAD 实验一致）
db_names, _ = load_multi_netvlad_descriptors(DB_H5_PATHS)
query_names, _ = load_multi_netvlad_descriptors(QUERY_H5_PATHS)

db_scene_ranges = get_db_scene_ranges(DB_H5_PATHS, db_names)
trajectory_ranges = get_query_trajectory_ranges(QUERY_H5_PATHS)

db_paths = get_all_image_paths(db_names, db_scene_ranges)
query_paths = get_all_image_paths(query_names, trajectory_ranges)

N_db = len(db_names)
n_queries = len(query_names)
print(f"DB: {N_db} 条, Query: {n_queries} 条")
print(f"示例 DB 路径: {db_paths[0]}")
print(f"示例 Query 路径: {query_paths[0]}")

DB: 11009 条, Query: 5988 条
示例 DB 路径: /home/lty/datasets/RealUAV/city1/tif_city1/100_856_1006.tif
示例 Query 路径: /home/lty/datasets/RealUAV/city1/tif_city1/001.jpg


## 2. 坐标法 GT 与 valid_for_metrics

In [7]:
gt_9_list, _ = build_gt_9_for_all_trajectories(
    trajectory_ranges, db_scene_ranges, db_names, datasets_base=DATASETS_BASE, verbose=True
)

valid = np.array([len(gt_9_list[i]) > 0 for i in range(n_queries)], dtype=bool)
exclude_scenes = {"industry2", "rural2"}
exclude_mask = np.zeros(n_queries, dtype=bool)
for scene_name, start, end in trajectory_ranges:
    if scene_name in exclude_scenes:
        exclude_mask[start:end] = True
valid_for_metrics = valid & ~exclude_mask

n_eval = int(valid_for_metrics.sum())
print(f"评估时有效 query 数（排除 industry2/rural2）: {n_eval}")

=== 坐标法 GT@25 各轨迹（未算出 GT 的会打印原因）===

--- 坐标法 GT 首帧诊断 [city1] ---
大图 mapbox geotransform (6 参数):
  gt[0] 左上角 X (投影/经度): 12123218.434142068
  gt[1] 像元宽:               0.2985821417389691
  gt[2] 旋转(常为 0):         0.0
  gt[3] 左上角 Y (投影/纬度): 4062398.742272254
  gt[4] 旋转(常为 0):         0.0
  gt[5] 像元高(常为负):       -0.2985821417389691
大图坐标系: EPSG:3857
第一张 query 图片的坐标 (uav_infos 第 1 行):
  经度 longitude = 108.91086
  纬度 latitude  = 34.24388
转换为大图投影坐标 (若大图非 WGS84 则先投影):
  x_geo = 12123901.477057505
  y_geo = 4061596.3503683605
逆 geotransform 得到大图像素坐标:
  px (列) = 2287.6214614151654
  py (行) = 2687.3405730843588
子图尺寸 (从 tif 目录任取一张读取):
  tile_w = 768, tile_h = 768
距离 query 坐标最近的子图左上角 startx, starty = 1906, 2356
距离 query 坐标最近的 25 张子图（按距离排序）对应的全局 DB 索引:
  [133, 134, 152, 153, 155, 156, 157, 173, 174, 175, 177, 178, 194, 195, 196, 197, 199, 215, 216, 217, 218, 219, 238, 239, 240]
对应的 db_names 示例 (前 3 个):
    db_names[133] = tif/220_1756_1906.tif
    db_names[134] = tif/221_1906_1906.tif
    db_names[152

## 3. ORB + 预训练词典 (orbvoc.dbow3)

使用你的 ORB 词典，需已安装 pyDBoW3。

In [8]:
ORB_VOCAB_PATH = "/home/lty/data/orbvoc.dbow3"
ORB_N_FEATURES = 2000

def recall_at_k(pred_I, gt_9_list, k, valid):
    nq = pred_I.shape[0]
    hit = 0
    count = 0
    for i in range(nq):
        if not valid[i] or len(gt_9_list[i]) == 0:
            continue
        count += 1
        gt_set = set(int(x) for x in gt_9_list[i])
        pred_topk = pred_I[i, :k]
        pred_valid = [int(p) for p in pred_topk if p >= 0]
        if any(p in gt_set for p in pred_valid):
            hit += 1
    return hit / count if count > 0 else 0.0

if HAS_PYDBOW3:
    print("加载 ORB 词典...")
    dbow = DBoW3Database(ORB_VOCAB_PATH, n_orb_features=ORB_N_FEATURES)
    print("添加 DB 图像...")
    t0 = time.perf_counter()
    dbow.add_all_from_paths(db_paths)
    t1 = time.perf_counter()
    print(f"  耗时 {t1-t0:.2f} s")
    print("Query 检索...")
    t0 = time.perf_counter()
    D_orb, I_orb = dbow.query_batch(query_paths, k=20)
    t2 = time.perf_counter()
    t_per_query_ms = (t2 - t0) / n_queries * 1000
    r1 = recall_at_k(I_orb, gt_9_list, 1, valid_for_metrics)
    r5 = recall_at_k(I_orb, gt_9_list, 5, valid_for_metrics)
    r10 = recall_at_k(I_orb, gt_9_list, 10, valid_for_metrics)
    r20 = recall_at_k(I_orb, gt_9_list, 20, valid_for_metrics)
    print("=== ORB + orbvoc.dbow3 检索性能 ===")
    print(f"  Recall@1  = {r1:.4f}")
    print(f"  Recall@5  = {r5:.4f}")
    print(f"  Recall@10 = {r10:.4f}")
    print(f"  Recall@20 = {r20:.4f}")
    print(f"  Query time = {t_per_query_ms:.4f} ms/query")
else:
    print("pyDBoW3 未安装，跳过 ORB 实验。安装方式: https://github.com/foxis/pyDBoW3")

pyDBoW3 未安装，跳过 ORB 实验。安装方式: https://github.com/foxis/pyDBoW3


## 4. SIFT + 自建词典

SIFT 无通用预训练词典，需从 DB 图像采样用 KMeans 自建。可调整 `N_VOCAB_SAMPLES`、`VOCAB_SIZE`、`SIFT_N_FEATURES`。

In [10]:
N_VOCAB_SAMPLES = 2000   # 用于构建词汇表的 DB 图像数
VOCAB_SIZE = 1024       # 视觉词数量
SIFT_N_FEATURES = 1500  # 每张图最多提取的 SIFT 特征数
SAMPLE_PER_IMAGE = 80   # 每张图采样多少特征参与 KMeans

np.random.seed(42)
vocab_idx = np.random.choice(N_db, min(N_VOCAB_SAMPLES, N_db), replace=False)

print("提取词汇表采样图像的特征...")
t0 = time.perf_counter()
desc_list = []
for i in vocab_idx:
    _, desc = extract_sift_features(db_paths[i], n_features=SIFT_N_FEATURES)
    desc_list.append(desc)
t1 = time.perf_counter()
print(f"  耗时 {t1-t0:.2f} s")

print("构建词汇表 (KMeans)...")
vocabulary = build_vocabulary(desc_list, vocab_size=VOCAB_SIZE, sample_per_image=SAMPLE_PER_IMAGE)
print(f"  词汇表大小: {VOCAB_SIZE}")

提取词汇表采样图像的特征...
  耗时 151.92 s
构建词汇表 (KMeans)...


RuntimeError: 需要 sklearn: pip install scikit-learn

## 5. 全库 DB 与 Query 的 BoW 向量 (SIFT)

In [ ]:
def extract_bow_batch(paths, vocabulary, n_features=SIFT_N_FEATURES):
    hists = []
    for i, p in enumerate(paths):
        if (i + 1) % 500 == 0 or i == 0:
            print(f"  处理 {i+1}/{len(paths)} ...", flush=True)
        _, desc = extract_sift_features(p, n_features=n_features)
        h = compute_bow_histogram(desc, vocabulary)
        hists.append(h)
    return np.array(hists, dtype=np.float32)

print("DB BoW 直方图...")
t0 = time.perf_counter()
db_hists = extract_bow_batch(db_paths, vocabulary)
t1 = time.perf_counter()
print(f"  耗时 {t1-t0:.2f} s")

print("Query BoW 直方图...")
t0 = time.perf_counter()
query_hists = extract_bow_batch(query_paths, vocabulary)
t2 = time.perf_counter()
print(f"  耗时 {t2-t0:.2f} s")

print("TF-IDF 加权...")
db_tfidf = compute_tfidf_vectors(db_hists)
query_tfidf = compute_tfidf_vectors(query_hists)
print(f"db_tfidf: {db_tfidf.shape}, query_tfidf: {query_tfidf.shape}")

## 6. SIFT 检索与 Recall 评估

In [ ]:
def recall_at_k(pred_I, gt_9_list, k, valid):
    nq = pred_I.shape[0]
    hit = 0
    count = 0
    for i in range(nq):
        if not valid[i] or len(gt_9_list[i]) == 0:
            continue
        count += 1
        gt_set = set(int(x) for x in gt_9_list[i])
        pred_topk = pred_I[i, :k]
        pred_valid = [int(p) for p in pred_topk if p >= 0]
        if any(p in gt_set for p in pred_valid):
            hit += 1
    return hit / count if count > 0 else 0.0

k_retrieve = 20
bow_db = BoWDatabase(db_tfidf)

t0 = time.perf_counter()
D_bow, I_bow = bow_db.query(query_tfidf, k=k_retrieve)
t_search = time.perf_counter() - t0
t_per_query_ms = (t_search / n_queries) * 1000

r1 = recall_at_k(I_bow, gt_9_list, 1, valid_for_metrics)
r5 = recall_at_k(I_bow, gt_9_list, 5, valid_for_metrics)
r10 = recall_at_k(I_bow, gt_9_list, 10, valid_for_metrics)
r20 = recall_at_k(I_bow, gt_9_list, 20, valid_for_metrics)

print("=== BoW (SIFT + TF-IDF) 检索性能 ===")
print(f"  Recall@1  = {r1:.4f}")
print(f"  Recall@5  = {r5:.4f}")
print(f"  Recall@10 = {r10:.4f}")
print(f"  Recall@20 = {r20:.4f}")
print(f"  Query time = {t_per_query_ms:.4f} ms/query")